In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import nibabel as nib

import tensorflow as tf

from scipy.ndimage import binary_fill_holes
from scipy.spatial.distance import cdist


print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s):", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(
                gpu,
                True
            )

        print("✓ GPU memory growth enabled")

    except RuntimeError as e:
        print("GPU memory-growth warning:", e)

TensorFlow: 2.20.0
GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU memory growth enabled


In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("✓ TensorFlow deterministic operations enabled")

except Exception as e:
    print("Determinism note:", e)


print("Seed:", SEED)

✓ TensorFlow deterministic operations enabled
Seed: 42


In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE1_DIR = (
    PROJECT_DIR /
    "phase1_segmentation"
)

P1A_DIR = (
    PHASE1_DIR /
    "phase1A_resolution"
)

P1B_DIR = (
    PHASE1_DIR /
    "phase1B_hyperparameters"
)

P1B_CONFIG_DIR = (
    P1B_DIR /
    "configs"
)

P1B_MODEL_DIR = (
    P1B_DIR /
    "models"
)

P1B_HISTORY_DIR = (
    P1B_DIR /
    "histories"
)

P1B_TABLE_DIR = (
    P1B_DIR /
    "tables"
)

P1B_PLOT_DIR = (
    P1B_DIR /
    "plots"
)


for folder in [
    P1B_CONFIG_DIR,
    P1B_MODEL_DIR,
    P1B_HISTORY_DIR,
    P1B_TABLE_DIR,
    P1B_PLOT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("Project:")
print(PROJECT_DIR)

print("\nPhase 1B:")
print(P1B_DIR)

print("\n✓ Paths ready.")

Project:
/content/drive/MyDrive/CAMUS_FINAL

Phase 1B:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters

✓ Paths ready.


In [ ]:
# ================================================================
# LOAD PHASE 1A RESOLUTION DECISION
# ================================================================

RESOLUTION_DECISION_PATH = (
    P1A_DIR /
    "configs" /
    "phase1A_final_resolution_decision.json"
)

assert RESOLUTION_DECISION_PATH.exists(), (
    "Phase 1A resolution decision file missing."
)

with open(
    RESOLUTION_DECISION_PATH,
    "r"
) as f:
    resolution_decision = json.load(f)

SELECTED_RESOLUTION = int(
    resolution_decision[
        "selected_resolution"
    ]
)


# ================================================================
# LOAD PHASE 1B-1 LEARNING RATE DECISION
# ================================================================

LR_DECISION_PATH = (
    P1B_CONFIG_DIR /
    "phase1B1_learning_rate_decision.json"
)

assert LR_DECISION_PATH.exists(), (
    "Phase 1B-1 learning-rate decision file missing."
)

with open(
    LR_DECISION_PATH,
    "r"
) as f:
    lr_decision = json.load(f)

SELECTED_LEARNING_RATE = float(
    lr_decision[
        "selected_learning_rate"
    ]
)


# ================================================================
# LOAD PHASE 1B-2 OPTIMIZER DECISION
# ================================================================

OPT_DECISION_PATH = (
    P1B_CONFIG_DIR /
    "phase1B2_optimizer_decision.json"
)

assert OPT_DECISION_PATH.exists(), (
    "Phase 1B-2 optimizer decision file missing."
)

with open(
    OPT_DECISION_PATH,
    "r"
) as f:
    optimizer_decision = json.load(f)

SELECTED_OPTIMIZER = (
    optimizer_decision[
        "selected_optimizer"
    ]
)


# ================================================================
# VERIFY EXPECTED VALUES
# ================================================================

assert SELECTED_RESOLUTION == 256

assert np.isclose(
    SELECTED_LEARNING_RATE,
    1e-4
)

assert SELECTED_OPTIMIZER == "Adam"


print("Loaded previous decisions:")

print(
    "Resolution    :",
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    "Learning rate :",
    SELECTED_LEARNING_RATE
)

print(
    "Optimizer     :",
    SELECTED_OPTIMIZER
)

print("\n✓ Previous decisions verified.")

Loaded previous decisions:
Resolution    : 256×256
Learning rate : 0.0001
Optimizer     : Adam

✓ Previous decisions verified.


In [ ]:
# CELL 6A — ONLY IF DATASET IS MISSING

import zipfile

OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR /
    "download"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)

DATASET_PATH = (
    INNER_EXTRACT_DIR /
    "database_nifti"
)


if not DATASET_PATH.exists():

    assert OUTER_ZIP_PATH.exists(), (
        f"Archive not found: {OUTER_ZIP_PATH}"
    )

    if not INNER_ARCHIVE_PATH.exists():

        print("Extracting outer archive...")

        OUTER_EXTRACT_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            OUTER_ZIP_PATH,
            "r"
        ) as archive:

            archive.extractall(
                OUTER_EXTRACT_DIR
            )

        print("✓ Outer archive extracted.")


    print("Extracting inner CAMUS archive...")

    INNER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        INNER_ARCHIVE_PATH,
        "r"
    ) as archive:

        archive.extractall(
            INNER_EXTRACT_DIR
        )


assert DATASET_PATH.exists()

print("\n✓ CAMUS dataset ready:")
print(DATASET_PATH)

Extracting outer archive...
✓ Outer archive extracted.
Extracting inner CAMUS archive...

✓ CAMUS dataset ready:
/content/camus_data_final/database_nifti


In [ ]:
SPLIT_PATH = (
    PROJECT_DIR /
    "splits" /
    "patient_split_seed42.json"
)

assert SPLIT_PATH.exists(), (
    f"Split file missing:\n{SPLIT_PATH}"
)


with open(
    SPLIT_PATH,
    "r"
) as f:
    patient_split = json.load(f)


train_ids = patient_split.get(
    "train_ids",
    patient_split.get("train")
)

val_ids = patient_split.get(
    "val_ids",
    patient_split.get(
        "validation",
        patient_split.get("val")
    )
)

test_ids = patient_split.get(
    "test_ids",
    patient_split.get("test")
)


assert train_ids is not None
assert val_ids is not None
assert test_ids is not None

assert len(train_ids) == 400
assert len(val_ids) == 50
assert len(test_ids) == 50

assert not (
    set(train_ids) & set(val_ids)
)

assert not (
    set(train_ids) & set(test_ids)
)

assert not (
    set(val_ids) & set(test_ids)
)


print("Train patients     :", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients      :", len(test_ids))

print("\n✓ Patient-level split verified.")
print("✓ Test set remains excluded.")

Train patients     : 400
Validation patients: 50
Test patients      : 50

✓ Patient-level split verified.
✓ Test set remains excluded.


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 8
# PRE-EXPERIMENT DESIGN VERIFICATION
# ================================================================

BASELINE_PARAMETER_COUNT = 1_940_817

BATCH_SIZES_TO_TEST = [
    2,
    4,
    8
]

MAX_EPOCHS = 30

THRESHOLD = 0.50

TRAIN_FRAMES = (
    len(train_ids) * 2
)

VAL_FRAMES = (
    len(val_ids) * 2
)


patient_folders = sorted([
    p
    for p in DATASET_PATH.iterdir()
    if p.is_dir()
    and p.name.startswith("patient")
])

assert len(patient_folders) == 500


print("=" * 70)
print("PHASE 1B-3 — CONTROLLED BATCH SIZE EXPERIMENT")
print("=" * 70)


print("\nLOCKED FROM PREVIOUS EXPERIMENTS")

print(
    "Resolution        :",
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    "Learning rate     :",
    SELECTED_LEARNING_RATE
)

print(
    "Optimizer         :",
    SELECTED_OPTIMIZER
)

print(
    "Architecture      : Baseline U-Net"
)

print(
    "Parameters        :",
    BASELINE_PARAMETER_COUNT
)

print(
    "Loss              : BCE + Dice"
)

print(
    "Augmentation      : None"
)

print(
    "Epoch budget      :",
    MAX_EPOCHS
)

print(
    "Threshold         :",
    THRESHOLD
)


print("\nVARIABLE BEING TESTED")

print(
    "Batch sizes       :",
    BATCH_SIZES_TO_TEST
)


print("\nDATA EXPOSURE")

print(
    "Training frames   :",
    TRAIN_FRAMES
)

print(
    "Validation frames :",
    VAL_FRAMES
)

print(
    "Epochs per run    :",
    MAX_EPOCHS
)

print(
    "Training examples seen per run:",
    TRAIN_FRAMES * MAX_EPOCHS
)


print("\nOPTIMIZER UPDATES PER EPOCH")

for batch_size in BATCH_SIZES_TO_TEST:

    steps = (
        TRAIN_FRAMES //
        batch_size
    )

    print(
        f"Batch {batch_size}: "
        f"{steps} updates/epoch, "
        f"{steps * MAX_EPOCHS} updates over 30 epochs"
    )


print("\nEXPERIMENTAL INTERPRETATION")

print(
    "Epoch count and total training-data exposure "
    "are held constant."
)

print(
    "The number of optimizer updates is allowed to "
    "change naturally with batch size."
)

print(
    "Batch size is therefore treated as the complete "
    "training configuration being investigated."
)


print("\nDATASET")

print(
    "Train patients     :",
    len(train_ids)
)

print(
    "Validation patients:",
    len(val_ids)
)

print(
    "Test patients      :",
    len(test_ids)
)


print(
    "\nTest used for batch-size selection: NO"
)

print(
    "\n✓ Phase 1B-3 experimental design verified."
)

print(
    "✓ Ready to create the controlled batch-size runs."
)

PHASE 1B-3 — CONTROLLED BATCH SIZE EXPERIMENT

LOCKED FROM PREVIOUS EXPERIMENTS
Resolution        : 256×256
Learning rate     : 0.0001
Optimizer         : Adam
Architecture      : Baseline U-Net
Parameters        : 1940817
Loss              : BCE + Dice
Augmentation      : None
Epoch budget      : 30
Threshold         : 0.5

VARIABLE BEING TESTED
Batch sizes       : [2, 4, 8]

DATA EXPOSURE
Training frames   : 800
Validation frames : 100
Epochs per run    : 30
Training examples seen per run: 24000

OPTIMIZER UPDATES PER EPOCH
Batch 2: 400 updates/epoch, 12000 updates over 30 epochs
Batch 4: 200 updates/epoch, 6000 updates over 30 epochs
Batch 8: 100 updates/epoch, 3000 updates over 30 epochs

EXPERIMENTAL INTERPRETATION
Epoch count and total training-data exposure are held constant.
The number of optimizer updates is allowed to change naturally with batch size.
Batch size is therefore treated as the complete training configuration being investigated.

DATASET
Train patients     : 400
V

In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 9
# CAMUS LOADING + PREPROCESSING
# ================================================================

def load_nifti_2d(file_path):

    nii = nib.load(str(file_path))

    data = np.asarray(
        nii.get_fdata(),
        dtype=np.float32
    )

    if data.ndim == 3 and data.shape[-1] == 1:
        data = data[:, :, 0]

    if data.ndim != 2:
        raise ValueError(
            f"Expected 2D image, got {data.shape}"
        )

    return data


def load_camus_frame(patient_id, phase):

    patient_dir = (
        DATASET_PATH /
        patient_id
    )

    image = load_nifti_2d(
        patient_dir /
        f"{patient_id}_4CH_{phase}.nii.gz"
    )

    full_mask = load_nifti_2d(
        patient_dir /
        f"{patient_id}_4CH_{phase}_gt.nii.gz"
    )

    # CAMUS label 1 = LV cavity
    lv_mask = (
        full_mask == 1
    ).astype(np.float32)

    return image, lv_mask


def preprocess_frame(
    image,
    mask,
    resolution=SELECTED_RESOLUTION
):

    image_resized = cv2.resize(
        image,
        (resolution, resolution),
        interpolation=cv2.INTER_LINEAR
    )

    mask_resized = cv2.resize(
        mask,
        (resolution, resolution),
        interpolation=cv2.INTER_NEAREST
    )

    minimum = float(
        image_resized.min()
    )

    maximum = float(
        image_resized.max()
    )

    if maximum > minimum:

        image_resized = (
            image_resized - minimum
        ) / (
            maximum - minimum
        )

    else:

        image_resized = np.zeros_like(
            image_resized,
            dtype=np.float32
        )

    image_resized = (
        image_resized.astype(
            np.float32
        )
    )

    mask_resized = (
        mask_resized > 0.5
    ).astype(np.float32)

    return (
        image_resized[..., None],
        mask_resized[..., None]
    )


print("✓ CAMUS loading and preprocessing ready.")

✓ CAMUS loading and preprocessing ready.


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 10
# DYNAMIC DATASET BUILDER
# ================================================================

def frame_generator(
    patient_ids,
    shuffle=True,
    seed=SEED
):

    frame_list = [
        (pid, phase)
        for pid in patient_ids
        for phase in ["ED", "ES"]
    ]

    rng = np.random.default_rng(
        seed
    )

    while True:

        indices = np.arange(
            len(frame_list)
        )

        if shuffle:
            rng.shuffle(indices)

        for index in indices:

            pid, phase = (
                frame_list[index]
            )

            image, mask = (
                load_camus_frame(
                    pid,
                    phase
                )
            )

            x, y = preprocess_frame(
                image,
                mask
            )

            yield x, y


OUTPUT_SIGNATURE = (

    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    ),

    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    )
)


def build_datasets(batch_size):

    train_dataset = (
        tf.data.Dataset
        .from_generator(
            lambda: frame_generator(
                train_ids,
                shuffle=True,
                seed=SEED
            ),
            output_signature=
                OUTPUT_SIGNATURE
        )
        .batch(
            batch_size,
            drop_remainder=True
        )
        .prefetch(1)
    )

    validation_dataset = (
        tf.data.Dataset
        .from_generator(
            lambda: frame_generator(
                val_ids,
                shuffle=False,
                seed=SEED
            ),
            output_signature=
                OUTPUT_SIGNATURE
        )
        .batch(
            batch_size,
            drop_remainder=False
        )
        .prefetch(1)
    )

    train_steps = (
        TRAIN_FRAMES //
        batch_size
    )

    validation_steps = int(
        np.ceil(
            VAL_FRAMES /
            batch_size
        )
    )

    return (
        train_dataset,
        validation_dataset,
        train_steps,
        validation_steps
    )


print("Dataset configurations:")

for batch_size in BATCH_SIZES_TO_TEST:

    _, _, train_steps, val_steps = (
        build_datasets(
            batch_size
        )
    )

    print(
        f"Batch {batch_size}: "
        f"train steps={train_steps}, "
        f"validation steps={val_steps}"
    )


print("\n✓ Dynamic dataset builder ready.")

Dataset configurations:
Batch 2: train steps=400, validation steps=50
Batch 4: train steps=200, validation steps=25
Batch 8: train steps=100, validation steps=13

✓ Dynamic dataset builder ready.


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 11
# FIXED LOSS + METRICS
# ================================================================

SMOOTH = 1e-6


def dice_coefficient(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
    )

    dice = (
        2.0 * intersection + SMOOTH
    ) / (
        denominator + SMOOTH
    )

    return tf.reduce_mean(
        dice
    )


def iou_metric(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    union = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
        -
        intersection
    )

    iou = (
        intersection + SMOOTH
    ) / (
        union + SMOOTH
    )

    return tf.reduce_mean(
        iou
    )


def dice_loss(
    y_true,
    y_pred
):

    return (
        1.0 -
        dice_coefficient(
            y_true,
            y_pred
        )
    )


def bce_dice_loss(
    y_true,
    y_pred
):

    bce = (
        tf.keras.losses
        .binary_crossentropy(
            y_true,
            y_pred
        )
    )

    bce = tf.reduce_mean(
        bce
    )

    return (
        bce +
        dice_loss(
            y_true,
            y_pred
        )
    )


print("✓ Fixed BCE + Dice loss ready.")
print("✓ Dice and IoU metrics ready.")

✓ Fixed BCE + Dice loss ready.
✓ Dice and IoU metrics ready.


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 12
# FIXED BASELINE U-NET
# ================================================================

def conv_block(
    x,
    filters
):

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    return x


def build_baseline_unet(
    input_shape=(256, 256, 1)
):

    inputs = tf.keras.Input(
        shape=input_shape
    )

    # Encoder
    c1 = conv_block(
        inputs,
        16
    )
    p1 = tf.keras.layers.MaxPooling2D()(c1)

    c2 = conv_block(
        p1,
        32
    )
    p2 = tf.keras.layers.MaxPooling2D()(c2)

    c3 = conv_block(
        p2,
        64
    )
    p3 = tf.keras.layers.MaxPooling2D()(c3)

    c4 = conv_block(
        p3,
        128
    )
    p4 = tf.keras.layers.MaxPooling2D()(c4)

    # Bottleneck
    c5 = conv_block(
        p4,
        256
    )

    # Decoder
    u6 = tf.keras.layers.Conv2DTranspose(
        128,
        2,
        strides=2,
        padding="same"
    )(c5)

    u6 = tf.keras.layers.Concatenate()(
        [u6, c4]
    )

    c6 = conv_block(
        u6,
        128
    )

    u7 = tf.keras.layers.Conv2DTranspose(
        64,
        2,
        strides=2,
        padding="same"
    )(c6)

    u7 = tf.keras.layers.Concatenate()(
        [u7, c3]
    )

    c7 = conv_block(
        u7,
        64
    )

    u8 = tf.keras.layers.Conv2DTranspose(
        32,
        2,
        strides=2,
        padding="same"
    )(c7)

    u8 = tf.keras.layers.Concatenate()(
        [u8, c2]
    )

    c8 = conv_block(
        u8,
        32
    )

    u9 = tf.keras.layers.Conv2DTranspose(
        16,
        2,
        strides=2,
        padding="same"
    )(c8)

    u9 = tf.keras.layers.Concatenate()(
        [u9, c1]
    )

    c9 = conv_block(
        u9,
        16
    )

    outputs = tf.keras.layers.Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c9)

    segmentation_model = (
        tf.keras.Model(
            inputs,
            outputs,
            name="Baseline_UNet"
        )
    )

    return segmentation_model


tf.keras.backend.clear_session()
gc.collect()

architecture_check_model = (
    build_baseline_unet(
        input_shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        )
    )
)

assert (
    architecture_check_model.count_params()
    ==
    BASELINE_PARAMETER_COUNT
)

print(
    "Parameter count:",
    architecture_check_model.count_params()
)

print("✓ Architecture verified.")

del architecture_check_model

tf.keras.backend.clear_session()
gc.collect()

Parameter count: 1940817
✓ Architecture verified.


0

In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 13
# COMMON MODEL INITIALIZATION
# ================================================================

BATCH_INITIAL_WEIGHTS_PATH = (
    P1B_CONFIG_DIR /
    "batch_size_experiment_initial_weights.weights.h5"
)


tf.keras.backend.clear_session()
gc.collect()

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


initialization_model = (
    build_baseline_unet(
        input_shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        )
    )
)


assert (
    initialization_model.count_params()
    ==
    BASELINE_PARAMETER_COUNT
)


initialization_model.save_weights(
    BATCH_INITIAL_WEIGHTS_PATH
)


print("Common initialization saved:")
print(BATCH_INITIAL_WEIGHTS_PATH)

print(
    "\nParameters:",
    initialization_model.count_params()
)


del initialization_model

tf.keras.backend.clear_session()
gc.collect()

print(
    "\n✓ All batch-size runs will "
    "start from identical weights."
)

Common initialization saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/batch_size_experiment_initial_weights.weights.h5

Parameters: 1940817

✓ All batch-size runs will start from identical weights.


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 14
# SAVE CONTROLLED EXPERIMENT PROTOCOL
# ================================================================

batch_protocol = {

    "experiment":
        "Phase 1B-3 — Controlled Batch Size Experiment",

    "status":
        "PROTOCOL FROZEN BEFORE TRAINING",

    "objective":
        (
            "Compare batch sizes 2, 4, and 8 while "
            "holding resolution, architecture, optimizer, "
            "learning rate, loss, augmentation, epoch budget, "
            "dataset split, and initialization fixed."
        ),

    "variable_tested":
        "batch_size",

    "batch_sizes":
        BATCH_SIZES_TO_TEST,

    "resolution":
        SELECTED_RESOLUTION,

    "learning_rate":
        SELECTED_LEARNING_RATE,

    "optimizer":
        SELECTED_OPTIMIZER,

    "architecture":
        "Baseline U-Net",

    "parameters":
        BASELINE_PARAMETER_COUNT,

    "loss":
        "BCE + Dice",

    "augmentation":
        False,

    "epochs":
        MAX_EPOCHS,

    "threshold":
        THRESHOLD,

    "seed":
        SEED,

    "same_initial_weights":
        True,

    "initial_weights_path":
        str(
            BATCH_INITIAL_WEIGHTS_PATH
        ),

    "train_patients":
        len(train_ids),

    "validation_patients":
        len(val_ids),

    "train_frames_per_epoch":
        TRAIN_FRAMES,

    "validation_frames":
        VAL_FRAMES,

    "total_training_frame_exposures":
        TRAIN_FRAMES * MAX_EPOCHS,

    "updates_per_epoch": {
        str(batch_size):
            TRAIN_FRAMES // batch_size
        for batch_size
        in BATCH_SIZES_TO_TEST
    },

    "comparison_policy":
        (
            "All configurations are trained for the same "
            "30 epochs and therefore receive the same total "
            "training-data exposure. Optimizer update count "
            "is allowed to vary naturally with batch size."
        ),

    "checkpoint_monitor":
        "val_dice_coefficient",

    "checkpoint_mode":
        "max",

    "early_stopping":
        False,

    "learning_rate_scheduler":
        False,

    "test_used":
        False
}


BATCH_PROTOCOL_PATH = (
    P1B_CONFIG_DIR /
    "batch_size_experiment_protocol.json"
)


with open(
    BATCH_PROTOCOL_PATH,
    "w"
) as f:

    json.dump(
        batch_protocol,
        f,
        indent=2
    )


print("=" * 70)
print("BATCH-SIZE PROTOCOL FROZEN")
print("=" * 70)

print("\nBatch sizes:")
print(BATCH_SIZES_TO_TEST)

print("\nFixed:")
print(
    f"Resolution    : "
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)
print(
    f"Learning rate : "
    f"{SELECTED_LEARNING_RATE}"
)
print(
    f"Optimizer     : "
    f"{SELECTED_OPTIMIZER}"
)
print(
    f"Epochs        : "
    f"{MAX_EPOCHS}"
)

print(
    "\nSame initialization: YES"
)

print(
    "Same frame exposure: YES"
)

print(
    "Same update count: NO "
    "(intentionally varies with batch size)"
)

print(
    "Test set used: NO"
)

print("\nSaved:")
print(BATCH_PROTOCOL_PATH)

print("\n✓ Protocol frozen.")

BATCH-SIZE PROTOCOL FROZEN

Batch sizes:
[2, 4, 8]

Fixed:
Resolution    : 256×256
Learning rate : 0.0001
Optimizer     : Adam
Epochs        : 30

Same initialization: YES
Same frame exposure: YES
Same update count: NO (intentionally varies with batch size)
Test set used: NO

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/batch_size_experiment_protocol.json

✓ Protocol frozen.


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 15
# BATCH-SIZE TRAINING FUNCTION
# ================================================================

def train_batch_size_experiment(
    batch_size
):

    assert (
        batch_size
        in BATCH_SIZES_TO_TEST
    )

    print("\n" + "=" * 70)
    print(
        f"TRAINING BATCH SIZE = {batch_size}"
    )
    print("=" * 70)

    # ------------------------------------------------------------
    # Clean memory
    # ------------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    # ------------------------------------------------------------
    # Reset reproducibility
    # ------------------------------------------------------------

    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    # ------------------------------------------------------------
    # Build dataset specifically for this batch size
    # ------------------------------------------------------------

    (
        training_dataset,
        validation_dataset,
        train_steps,
        validation_steps
    ) = build_datasets(
        batch_size
    )

    print(
        "Training steps/epoch:",
        train_steps
    )

    print(
        "Validation steps:",
        validation_steps
    )

    # ------------------------------------------------------------
    # Fresh model
    # ------------------------------------------------------------

    segmentation_model = (
        build_baseline_unet(
            input_shape=(
                SELECTED_RESOLUTION,
                SELECTED_RESOLUTION,
                1
            )
        )
    )

    # Same initialization for every run
    segmentation_model.load_weights(
        BATCH_INITIAL_WEIGHTS_PATH
    )

    assert (
        segmentation_model.count_params()
        ==
        BASELINE_PARAMETER_COUNT
    )

    # ------------------------------------------------------------
    # Fixed Adam optimizer
    # ------------------------------------------------------------

    optimizer_instance = (
        tf.keras.optimizers.Adam(
            learning_rate=
                SELECTED_LEARNING_RATE
        )
    )

    segmentation_model.compile(
        optimizer=optimizer_instance,
        loss=bce_dice_loss,
        metrics=[
            dice_coefficient,
            iou_metric
        ]
    )

    # ------------------------------------------------------------
    # Output paths
    # ------------------------------------------------------------

    model_path = (
        P1B_MODEL_DIR /
        f"batch_{batch_size}_best.keras"
    )

    history_path = (
        P1B_HISTORY_DIR /
        f"batch_{batch_size}_history.csv"
    )

    # ------------------------------------------------------------
    # Checkpoint
    # ------------------------------------------------------------

    checkpoint = (
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(
                model_path
            ),
            monitor=
                "val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    )

    # ------------------------------------------------------------
    # Train
    # ------------------------------------------------------------

    start_time = time.time()

    history = segmentation_model.fit(
        training_dataset,
        validation_data=
            validation_dataset,
        epochs=MAX_EPOCHS,
        steps_per_epoch=
            train_steps,
        validation_steps=
            validation_steps,
        callbacks=[
            checkpoint
        ],
        verbose=1
    )

    training_minutes = (
        time.time() -
        start_time
    ) / 60.0

    # ------------------------------------------------------------
    # Save history
    # ------------------------------------------------------------

    history_df = pd.DataFrame(
        history.history
    )

    history_df.index = (
        np.arange(
            len(history_df)
        ) + 1
    )

    history_df.index.name = "epoch"

    history_df.to_csv(
        history_path
    )

    # ------------------------------------------------------------
    # Best epoch
    # ------------------------------------------------------------

    val_dice_values = np.asarray(
        history.history[
            "val_dice_coefficient"
        ],
        dtype=float
    )

    best_index = int(
        np.argmax(
            val_dice_values
        )
    )

    best_epoch = (
        best_index + 1
    )

    result = {

        "batch_size":
            int(batch_size),

        "learning_rate":
            float(
                SELECTED_LEARNING_RATE
            ),

        "optimizer":
            SELECTED_OPTIMIZER,

        "parameters":
            int(
                segmentation_model.count_params()
            ),

        "epochs":
            MAX_EPOCHS,

        "steps_per_epoch":
            int(train_steps),

        "total_optimizer_updates":
            int(
                train_steps *
                MAX_EPOCHS
            ),

        "best_epoch":
            int(best_epoch),

        "best_val_dice_training":
            float(
                val_dice_values[
                    best_index
                ]
            ),

        "val_iou_at_best_dice_epoch":
            float(
                history.history[
                    "val_iou_metric"
                ][best_index]
            ),

        "training_minutes":
            float(
                training_minutes
            ),

        "model_path":
            str(model_path),

        "history_path":
            str(history_path)
    }


    print("\n" + "-" * 70)

    print(
        f"BATCH {batch_size} COMPLETE"
    )

    print(
        "Best epoch:",
        result["best_epoch"]
    )

    print(
        "Best validation Dice:",
        round(
            result[
                "best_val_dice_training"
            ],
            6
        )
    )

    print(
        "Validation IoU:",
        round(
            result[
                "val_iou_at_best_dice_epoch"
            ],
            6
        )
    )

    print(
        "Total optimizer updates:",
        result[
            "total_optimizer_updates"
        ]
    )

    print(
        "Training time:",
        round(
            training_minutes,
            2
        ),
        "minutes"
    )

    print(
        "Checkpoint:",
        model_path
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # Clean memory before next run
    # ------------------------------------------------------------

    del segmentation_model
    del optimizer_instance
    del history
    del training_dataset
    del validation_dataset

    tf.keras.backend.clear_session()
    gc.collect()

    return result

In [ ]:
batch_2_result = (
    train_batch_size_experiment(
        2
    )
)

batch_2_result


TRAINING BATCH SIZE = 2
Training steps/epoch: 400
Validation steps: 50
Epoch 1/30
399/400 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - dice_coefficient: 0.2679 - iou_metric: 0.1688 - loss: 1.1402
Epoch 1: val_dice_coefficient improved from None to 0.72898, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_2_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_2_best.keras
400/400 ━━━━━━━━━━━━━━━━━━━━ 32s 60ms/step - dice_coefficient: 0.4340 - iou_metric: 0.3009 - loss: 0.8232 - val_dice_coefficient: 0.7290 - val_iou_metric: 0.5783 - val_loss: 0.3932
Epoch 2/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - dice_coefficient: 0.7472 - iou_metric: 0.6056 - loss: 0.3714
Epoch 2: val_dice_coefficient improved from 0.72898 to 0.83560, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_2_best.keras

Epoch 2

{'batch_size': 2,
 'learning_rate': 0.0001,
 'optimizer': 'Adam',
 'parameters': 1940817,
 'epochs': 30,
 'steps_per_epoch': 400,
 'total_optimizer_updates': 12000,
 'best_epoch': 28,
 'best_val_dice_training': 0.9300199747085571,
 'val_iou_at_best_dice_epoch': 0.8713198900222778,
 'training_minutes': 12.317301615079243,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_2_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/histories/batch_2_history.csv'}

In [ ]:
batch_4_result = (
    train_batch_size_experiment(
        4
    )
)

batch_4_result


TRAINING BATCH SIZE = 4
Training steps/epoch: 200
Validation steps: 25
Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - dice_coefficient: 0.1832 - iou_metric: 0.1053 - loss: 1.3305
Epoch 1: val_dice_coefficient improved from None to 0.43977, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_4_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_4_best.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 29s 105ms/step - dice_coefficient: 0.2759 - iou_metric: 0.1687 - loss: 1.0598 - val_dice_coefficient: 0.4398 - val_iou_metric: 0.2837 - val_loss: 0.7019
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - dice_coefficient: 0.5634 - iou_metric: 0.4030 - loss: 0.6001
Epoch 2: val_dice_coefficient improved from 0.43977 to 0.72617, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_4_best.keras

Epoch 

{'batch_size': 4,
 'learning_rate': 0.0001,
 'optimizer': 'Adam',
 'parameters': 1940817,
 'epochs': 30,
 'steps_per_epoch': 200,
 'total_optimizer_updates': 6000,
 'best_epoch': 30,
 'best_val_dice_training': 0.9254956245422363,
 'val_iou_at_best_dice_epoch': 0.8635912537574768,
 'training_minutes': 9.984476792812348,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_4_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/histories/batch_4_history.csv'}

In [ ]:
batch_8_result = (
    train_batch_size_experiment(
        8
    )
)

batch_8_result


TRAINING BATCH SIZE = 8
Training steps/epoch: 100
Validation steps: 13
Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.1431 - iou_metric: 0.0781 - loss: 1.4847
Epoch 1: val_dice_coefficient improved from None to 0.34929, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_8_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_8_best.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 30s 222ms/step - dice_coefficient: 0.1484 - iou_metric: 0.0822 - loss: 1.3485 - val_dice_coefficient: 0.3493 - val_iou_metric: 0.2135 - val_loss: 0.8307
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - dice_coefficient: 0.3882 - iou_metric: 0.2435 - loss: 0.7897
Epoch 2: val_dice_coefficient improved from 0.34929 to 0.44705, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_8_best.keras

Epoc

{'batch_size': 8,
 'learning_rate': 0.0001,
 'optimizer': 'Adam',
 'parameters': 1940817,
 'epochs': 30,
 'steps_per_epoch': 100,
 'total_optimizer_updates': 3000,
 'best_epoch': 29,
 'best_val_dice_training': 0.9193903207778931,
 'val_iou_at_best_dice_epoch': 0.8536010384559631,
 'training_minutes': 9.602443393071493,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/batch_8_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/histories/batch_8_history.csv'}

In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 19
# TRAINING SUMMARY
# ================================================================

batch_training_summary = (
    pd.DataFrame([
        batch_2_result,
        batch_4_result,
        batch_8_result
    ])
)


BATCH_TRAINING_SUMMARY_PATH = (
    P1B_TABLE_DIR /
    "batch_size_training_summary.csv"
)


batch_training_summary.to_csv(
    BATCH_TRAINING_SUMMARY_PATH,
    index=False
)


display(
    batch_training_summary[
        [
            "batch_size",
            "parameters",
            "steps_per_epoch",
            "total_optimizer_updates",
            "best_epoch",
            "best_val_dice_training",
            "val_iou_at_best_dice_epoch",
            "training_minutes"
        ]
    ].round(6)
)


print("\n✓ Training summary saved:")
print(BATCH_TRAINING_SUMMARY_PATH)

,batch_size,parameters,steps_per_epoch,total_optimizer_updates,best_epoch,best_val_dice_training,val_iou_at_best_dice_epoch,training_minutes
0,2,1940817,400,12000,28,0.930020,0.871320,12.317302
1,4,1940817,200,6000,30,0.925496,0.863591,9.984477
2,8,1940817,100,3000,29,0.919390,0.853601,9.602443



✓ Training summary saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/batch_size_training_summary.csv


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 20
# POSTPROCESSING + VALIDATION METRICS
# ================================================================

def keep_largest_component(mask):

    mask = np.asarray(mask).astype(np.uint8)

    if mask.sum() == 0:
        return np.zeros_like(mask, dtype=bool)

    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    if n_labels <= 1:
        return np.zeros_like(mask, dtype=bool)

    largest_label = (
        1 +
        np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )
    )

    return labels == largest_label


def postprocess_prediction(mask):

    largest = keep_largest_component(mask)

    filled = binary_fill_holes(
        largest
    )

    return np.asarray(
        filled,
        dtype=bool
    )


def binary_dice(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    denominator = gt.sum() + pred.sum()

    if denominator == 0:
        return 1.0

    return (
        2.0 * intersection /
        denominator
    )


def binary_iou(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    union = np.logical_or(
        gt, pred
    ).sum()

    if union == 0:
        return 1.0

    return intersection / union


def binary_precision(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(gt, pred).sum()
    fp = np.logical_and(~gt, pred).sum()

    if (tp + fp) == 0:
        return 1.0 if gt.sum() == 0 else 0.0

    return tp / (tp + fp)


def binary_recall(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(gt, pred).sum()
    fn = np.logical_and(gt, ~pred).sum()

    if (tp + fn) == 0:
        return 1.0

    return tp / (tp + fn)


def boundary_points(mask):

    mask = np.asarray(mask).astype(np.uint8)

    kernel = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    eroded = cv2.erode(
        mask,
        kernel,
        iterations=1
    )

    boundary = (
        mask.astype(bool)
        &
        ~eroded.astype(bool)
    )

    return np.column_stack(
        np.where(boundary)
    )


def hd95_pixels(gt, pred):

    gt_points = boundary_points(gt)
    pred_points = boundary_points(pred)

    if (
        len(gt_points) == 0
        or len(pred_points) == 0
    ):
        return np.nan

    distances = cdist(
        gt_points,
        pred_points
    )

    gt_to_pred = distances.min(
        axis=1
    )

    pred_to_gt = distances.min(
        axis=0
    )

    all_distances = np.concatenate([
        gt_to_pred,
        pred_to_gt
    ])

    return float(
        np.percentile(
            all_distances,
            95
        )
    )


print("✓ Postprocessing and metrics ready.")

✓ Postprocessing and metrics ready.


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 21
# CHECKPOINT EVALUATION FUNCTION
# ================================================================

def evaluate_batch_checkpoint(
    batch_size
):

    model_path = (
        P1B_MODEL_DIR /
        f"batch_{batch_size}_best.keras"
    )

    assert model_path.exists(), (
        f"Checkpoint missing:\n{model_path}"
    )

    print("\n" + "=" * 70)
    print(
        f"EVALUATING BATCH SIZE = {batch_size}"
    )
    print("=" * 70)

    tf.keras.backend.clear_session()
    gc.collect()

    evaluation_model = (
        tf.keras.models.load_model(
            model_path,
            custom_objects={
                "bce_dice_loss":
                    bce_dice_loss,

                "dice_coefficient":
                    dice_coefficient,

                "iou_metric":
                    iou_metric
            }
        )
    )

    records = []

    for patient_number, pid in enumerate(
        val_ids,
        start=1
    ):

        for phase in ["ED", "ES"]:

            image, gt_mask = (
                load_camus_frame(
                    pid,
                    phase
                )
            )

            x, gt_resized = (
                preprocess_frame(
                    image,
                    gt_mask
                )
            )

            probability = (
                evaluation_model(
                    x[None, ...],
                    training=False
                )
                .numpy()[0, :, :, 0]
            )

            pred_raw = (
                probability >= THRESHOLD
            )

            pred_mask = (
                postprocess_prediction(
                    pred_raw
                )
            )

            gt_binary = (
                gt_resized[:, :, 0]
                > 0.5
            )

            gt_area = int(
                gt_binary.sum()
            )

            pred_area = int(
                pred_mask.sum()
            )

            if gt_area > 0:

                signed_area_error_pct = (
                    (pred_area - gt_area)
                    / gt_area
                    * 100.0
                )

                abs_area_error_pct = abs(
                    signed_area_error_pct
                )

            else:

                signed_area_error_pct = np.nan
                abs_area_error_pct = np.nan


            hd95 = hd95_pixels(
                gt_binary,
                pred_mask
            )


            records.append({

                "patient_id":
                    pid,

                "phase":
                    phase,

                "batch_size":
                    batch_size,

                "dice":
                    binary_dice(
                        gt_binary,
                        pred_mask
                    ),

                "iou":
                    binary_iou(
                        gt_binary,
                        pred_mask
                    ),

                "precision":
                    binary_precision(
                        gt_binary,
                        pred_mask
                    ),

                "recall":
                    binary_recall(
                        gt_binary,
                        pred_mask
                    ),

                "hd95_pixels":
                    hd95,

                "normalized_hd95":
                    (
                        hd95 /
                        SELECTED_RESOLUTION
                        if np.isfinite(hd95)
                        else np.nan
                    ),

                "gt_area_pixels":
                    gt_area,

                "pred_area_pixels":
                    pred_area,

                "signed_area_error_pct":
                    signed_area_error_pct,

                "abs_area_error_pct":
                    abs_area_error_pct
            })


        if patient_number % 10 == 0:

            print(
                f"Evaluated "
                f"{patient_number}/"
                f"{len(val_ids)} patients"
            )


    results_df = pd.DataFrame(
        records
    )

    assert len(results_df) == 100


    output_path = (
        P1B_TABLE_DIR /
        f"batch_{batch_size}_validation_metrics.csv"
    )

    results_df.to_csv(
        output_path,
        index=False
    )


    print(
        "\nFrames evaluated:",
        len(results_df)
    )

    print("Saved:")
    print(output_path)


    del evaluation_model

    tf.keras.backend.clear_session()
    gc.collect()

    return results_df

In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 22
# EVALUATE ALL BATCH SIZES
# ================================================================

batch_2_eval = (
    evaluate_batch_checkpoint(
        2
    )
)

batch_4_eval = (
    evaluate_batch_checkpoint(
        4
    )
)

batch_8_eval = (
    evaluate_batch_checkpoint(
        8
    )
)


assert len(batch_2_eval) == 100
assert len(batch_4_eval) == 100
assert len(batch_8_eval) == 100


print("\n✓ All batch-size checkpoints evaluated.")


EVALUATING BATCH SIZE = 2
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/batch_2_validation_metrics.csv

EVALUATING BATCH SIZE = 4
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/batch_4_validation_metrics.csv

EVALUATING BATCH SIZE = 8
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/batch_8_validation_metrics.csv

✓ All batch-size checkpoints evaluated.


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 23
# FINAL VALIDATION COMPARISON
# ================================================================

def summarize_batch_results(
    evaluation_df,
    training_result
):

    ed = evaluation_df[
        evaluation_df["phase"] == "ED"
    ]

    es = evaluation_df[
        evaluation_df["phase"] == "ES"
    ]


    return {

        "batch_size":
            training_result[
                "batch_size"
            ],

        "parameters":
            training_result[
                "parameters"
            ],

        "best_epoch":
            training_result[
                "best_epoch"
            ],

        "steps_per_epoch":
            training_result[
                "steps_per_epoch"
            ],

        "total_optimizer_updates":
            training_result[
                "total_optimizer_updates"
            ],

        "training_minutes":
            training_result[
                "training_minutes"
            ],

        "overall_dice":
            evaluation_df[
                "dice"
            ].mean(),

        "ED_dice":
            ed[
                "dice"
            ].mean(),

        "ES_dice":
            es[
                "dice"
            ].mean(),

        "overall_iou":
            evaluation_df[
                "iou"
            ].mean(),

        "precision":
            evaluation_df[
                "precision"
            ].mean(),

        "recall":
            evaluation_df[
                "recall"
            ].mean(),

        "normalized_hd95":
            evaluation_df[
                "normalized_hd95"
            ].mean(),

        "ED_area_MAE_pct":
            ed[
                "abs_area_error_pct"
            ].mean(),

        "ES_area_MAE_pct":
            es[
                "abs_area_error_pct"
            ].mean(),

        "overall_area_MAE_pct":
            evaluation_df[
                "abs_area_error_pct"
            ].mean()
    }


batch_comparison = pd.DataFrame([

    summarize_batch_results(
        batch_2_eval,
        batch_2_result
    ),

    summarize_batch_results(
        batch_4_eval,
        batch_4_result
    ),

    summarize_batch_results(
        batch_8_eval,
        batch_8_result
    )
])


BATCH_COMPARISON_PATH = (
    P1B_TABLE_DIR /
    "batch_size_validation_comparison.csv"
)


batch_comparison.to_csv(
    BATCH_COMPARISON_PATH,
    index=False
)


display(
    batch_comparison.round(4)
)


print("\n✓ Final batch-size comparison saved:")
print(BATCH_COMPARISON_PATH)

,batch_size,parameters,best_epoch,steps_per_epoch,total_optimizer_updates,training_minutes,overall_dice,ED_dice,ES_dice,overall_iou,precision,recall,normalized_hd95,ED_area_MAE_pct,ES_area_MAE_pct,overall_area_MAE_pct
0,2,1940817,28,400,12000,12.3173,0.9335,0.9435,0.9234,0.8774,0.9348,0.9378,0.0226,6.3140,10.3131,8.3135
1,4,1940817,30,200,6000,9.9845,0.9295,0.9401,0.9188,0.8705,0.9312,0.9334,0.0234,6.4293,10.0577,8.2435
2,8,1940817,29,100,3000,9.6024,0.9256,0.9387,0.9125,0.8645,0.9119,0.9464,0.0245,6.8319,12.9176,9.8748



✓ Final batch-size comparison saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/batch_size_validation_comparison.csv


In [ ]:
# ================================================================
# PHASE 1B-3 — CELL 24
# FINALIZE BATCH-SIZE SELECTION
# ================================================================

SELECTED_BATCH_SIZE = 2

selected_row = batch_comparison[
    batch_comparison["batch_size"] == SELECTED_BATCH_SIZE
].iloc[0]


batch_decision = {

    "experiment":
        "Phase 1B-3 — Controlled Batch Size Experiment",

    "status":
        "COMPLETED",

    "selected_batch_size":
        SELECTED_BATCH_SIZE,

    "batch_sizes_compared":
        [2, 4, 8],

    "fixed_resolution":
        SELECTED_RESOLUTION,

    "fixed_learning_rate":
        SELECTED_LEARNING_RATE,

    "fixed_optimizer":
        SELECTED_OPTIMIZER,

    "fixed_architecture":
        "Baseline U-Net",

    "parameter_count":
        BASELINE_PARAMETER_COUNT,

    "fixed_loss":
        "BCE + Dice",

    "augmentation":
        False,

    "epochs":
        MAX_EPOCHS,

    "threshold":
        THRESHOLD,

    "selection_dataset":
        "Validation set only",

    "test_set_used":
        False,

    "selected_validation_results": {

        "overall_dice":
            float(selected_row["overall_dice"]),

        "ED_dice":
            float(selected_row["ED_dice"]),

        "ES_dice":
            float(selected_row["ES_dice"]),

        "overall_iou":
            float(selected_row["overall_iou"]),

        "precision":
            float(selected_row["precision"]),

        "recall":
            float(selected_row["recall"]),

        "normalized_hd95":
            float(selected_row["normalized_hd95"]),

        "ED_area_MAE_pct":
            float(selected_row["ED_area_MAE_pct"]),

        "ES_area_MAE_pct":
            float(selected_row["ES_area_MAE_pct"]),

        "overall_area_MAE_pct":
            float(selected_row["overall_area_MAE_pct"])
    },

    "selection_reason": (
        "Batch size 2 was selected because it achieved the strongest "
        "overall segmentation performance under the predefined fixed "
        "30-epoch and equal-data-exposure protocol. It obtained the "
        "highest overall Dice, ED Dice, ES Dice, and IoU, as well as "
        "the lowest normalized HD95 and lowest ED area MAE. Batch size "
        "4 produced marginally lower ES and overall area MAE, but these "
        "differences were small, while batch size 2 showed consistently "
        "stronger overlap and boundary performance."
    ),

    "important_interpretation": (
        "The batch-size experiment held epoch count and total training "
        "data exposure constant, not optimizer-update count. Batch size "
        "2 therefore received more optimizer updates than batch sizes "
        "4 and 8. The result should be interpreted as batch size 2 "
        "performing best under this predefined training protocol, not "
        "as evidence that smaller batch size is inherently superior "
        "independent of update count."
    ),

    "optimizer_updates": {
        "batch_2": 12000,
        "batch_4": 6000,
        "batch_8": 3000
    },

    "next_experiment":
        "Phase 1B-4 — Controlled Augmentation Experiment"
}


BATCH_DECISION_PATH = (
    P1B_CONFIG_DIR /
    "phase1B3_batch_size_decision.json"
)


with open(
    BATCH_DECISION_PATH,
    "w"
) as f:

    json.dump(
        batch_decision,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 1B-3 COMPLETED")
print("=" * 70)

print("\nSelected batch size:")
print(SELECTED_BATCH_SIZE)

print("\nLOCKED SETTINGS")
print(
    f"Resolution    : "
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)
print(
    f"Learning rate : "
    f"{SELECTED_LEARNING_RATE}"
)
print(
    f"Optimizer     : "
    f"{SELECTED_OPTIMIZER}"
)
print(
    f"Batch size    : "
    f"{SELECTED_BATCH_SIZE}"
)

print("\nSelected validation performance:")
print(
    f"Overall Dice    : "
    f"{selected_row['overall_dice']:.4f}"
)
print(
    f"ED Dice         : "
    f"{selected_row['ED_dice']:.4f}"
)
print(
    f"ES Dice         : "
    f"{selected_row['ES_dice']:.4f}"
)
print(
    f"Overall IoU     : "
    f"{selected_row['overall_iou']:.4f}"
)
print(
    f"Normalized HD95 : "
    f"{selected_row['normalized_hd95']:.4f}"
)
print(
    f"ED Area MAE     : "
    f"{selected_row['ED_area_MAE_pct']:.2f}%"
)
print(
    f"ES Area MAE     : "
    f"{selected_row['ES_area_MAE_pct']:.2f}%"
)
print(
    f"Overall Area MAE: "
    f"{selected_row['overall_area_MAE_pct']:.2f}%"
)

print("\nTest set used: NO")

print("\nNext:")
print("Phase 1B-4 — Controlled Augmentation Experiment")

print("\nDecision saved:")
print(BATCH_DECISION_PATH)

print("\n✓ Batch size officially locked at 2.")

PHASE 1B-3 COMPLETED

Selected batch size:
2

LOCKED SETTINGS
Resolution    : 256×256
Learning rate : 0.0001
Optimizer     : Adam
Batch size    : 2

Selected validation performance:
Overall Dice    : 0.9335
ED Dice         : 0.9435
ES Dice         : 0.9234
Overall IoU     : 0.8774
Normalized HD95 : 0.0226
ED Area MAE     : 6.31%
ES Area MAE     : 10.31%
Overall Area MAE: 8.31%

Test set used: NO

Next:
Phase 1B-4 — Controlled Augmentation Experiment

Decision saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/phase1B3_batch_size_decision.json

✓ Batch size officially locked at 2.
